# Migração Oracle → ClickHouse via Spark

**Estratégia**: 
- Leitura Oracle via Spark JDBC
- Remoção de duplicatas no Spark DataFrame
- Gravação ClickHouse via Spark JDBC
- Tracking de progresso incremental

**IMPORTANTE**: Execute as células na ordem.

## 1. Criar Spark Session

In [ ]:
from pyspark.sql import SparkSession

spark = (
 SparkSession.builder
 .appName("oracle-clickhouse-migration")
 .config("spark.sql.shuffle.partitions", "8")
 .config("spark.driver.memory", "4g")
 .config("spark.executor.memory", "4g")
 .config("spark.sql.adaptive.enabled", "true")
 .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
 .config("spark.sql.execution.arrow.pyspark.enabled", "false")
 .getOrCreate()
)

print("Spark Session criada")

## 2. Configurações de Conexão

In [ ]:
# Oracle
oracle_host = "10.255.150.11"
oracle_port = 1521
oracle_service = "bi.grupotracker.com.br"
oracle_user = "clickhouse"
oracle_password = "qiU!EOoe"

jdbc_url = f"jdbc:oracle:thin:@//{oracle_host}:{oracle_port}/{oracle_service}"
jdbc_opts = {
 "url": jdbc_url,
 "user": oracle_user,
 "password": oracle_password,
 "driver": "oracle.jdbc.OracleDriver"
}

# ClickHouse
clickhouse_host = "e1a1lieug8.us-central1.gcp.clickhouse.cloud"
clickhouse_port = 8443
clickhouse_user = "default"
clickhouse_password = "_uv765EvWphL_"
clickhouse_database = "raw"

clickhouse_jdbc_url = f"jdbc:clickhouse://{clickhouse_host}:{clickhouse_port}/{clickhouse_database}?ssl=true"

print(f"Oracle: {jdbc_url}")
print(f"ClickHouse: {clickhouse_jdbc_url}")

## 3. Testar Conexões

In [ ]:
# Teste Oracle
df_test = spark.read.format("jdbc") \
 .option("url", jdbc_url) \
 .option("user", oracle_user) \
 .option("password", oracle_password) \
 .option("driver", "oracle.jdbc.OracleDriver") \
 .option("query", "SELECT 1 AS ok FROM dual") \
 .load()

df_test.show()
print("Oracle OK")

# Cliente ClickHouse (para gerenciar progresso)
import clickhouse_connect

client = clickhouse_connect.get_client(
 host=clickhouse_host,
 port=8443,
 username=clickhouse_user,
 password=clickhouse_password,
 database=clickhouse_database,
 secure=True
)
print("ClickHouse OK")

## 4. Definir Tabelas

In [ ]:
ch_tables = {
 "ginf_depara_cliente": "ginf.depara_cliente",
 "ginf_base_cep_completa": "ginf.BASE_CEP_COMPLETA",
 "bistage_tst_contratos_bi": "bistage.TST_CONTRATOS_BI",
 "siga_sc5030": "siga.SC5030",
 "siga_sc6030": "siga.SC6030",
 "ginf_tst_historico_solicitacoes": "ginf.TST_HISTORICO_SOLICITACOES",
 "ginf_tst_solicit_cadastradas": "ginf.TST_SOLICIT_CADASTRADAS",
 "siga_sd2030": "siga.SD2030",
 "siga_sf2030": "siga.SF2030",
 "siga_ztx030": "siga.ZTX030",
 "ginf_tst_contratos": "ginf.TST_CONTRATOS"
}

print(f"Total: {len(ch_tables)} tabelas")
for ch_tbl, oracle_tbl in ch_tables.items():
 print(f" {oracle_tbl} -> {ch_tbl}")

## 5. Migração Inicial (Primeiro Batch)

Execute esta célula para fazer a primeira carga.

In [ ]:
# Ver arquivo: oracle-clickhouse-migration-clean.ipynb Cell #14

## 6. Migração Incremental

Execute esta célula repetidamente até completar todas as tabelas.

In [ ]:
# Ver arquivo: oracle-clickhouse-migration-clean.ipynb Cell #16

## 7. Resetar Erros (Se necessário)

In [ ]:
# Ver arquivo: oracle-clickhouse-migration-clean.ipynb Cell #17

## 8. Monitorar Progresso

In [ ]:
progress = client.query_df("""
 SELECT 
 oracle_table,
 ch_table,
 rows_collected,
 total_rows,
 rows_remaining,
 ROUND(rows_collected / total_rows * 100, 2) as pct_complete,
 status,
 last_id
 FROM migration_progress FINAL
 ORDER BY pct_complete DESC
""")

print("\nPROGRESSO DA MIGRAÇÃO")
print("=" * 100)
for _, row in progress.iterrows():
 status_icon = {' 'partial': '', 'complete': '', 'error': '', 'pending': '⏳'}.get(row['status'], '')
 print(f"{status_icon} {row['oracle_table']:45s} {row['rows_collected']:>12,.0f} / {row['total_rows']:>12,.0f} ({row['pct_complete']:>6.2f}%)")

total_collected = progress['rows_collected'].sum()
total_rows = progress['total_rows'].sum()
pct_overall = (total_collected / total_rows * 100) if total_rows > 0 else 0

print("=" * 100)
print(f"TOTAL: {total_collected:,.0f} / {total_rows:,.0f} ({pct_overall:.2f}%)")